# 04 — Final Comparative Analysis: Linear vs Logistic Regression

## Layout Convention
Every plot is a **2 × 3 grid of heatmaps**:
- **Rows**: Linear regression (top) vs Logistic regression (bottom)
- **Columns**: CART, NORM, PMM

Each cell contains a **3 × 3 heatmap** indexed by ρ (rows) × σ² (columns),
showing the metric aggregated over all M iterations.

One such figure is produced for every combination of
**var_type** ∈ {Binary, Continuous} × **N** ∈ {100, 1000} × **p** ∈ {2, 5, 10}  →  12 figures per plot.

In [ ]:
# ── Imports & Config ─────────────────────────────────────────────────────────
import os, gc, json, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)

# ── Load simulation config ──────────────────────────────────────────────────
config_path = os.path.join("..", "config", "config.json")
with open(config_path) as f:
    config = json.load(f)

N_VALS      = sorted(config["simulation"]["N"])
P_VALS      = sorted(config["simulation"]["p"])
VAR_TYPES   = sorted(config["simulation"]["var_type"])
METHODS     = sorted(config["synthesis"]["methods"])
RHO_VALS    = sorted([float(r) for r in config["parameters"]["rho"]])
SIGMA2_VALS = sorted([float(s) for s in config["parameters"]["sigma_2"]])
BETA_TRUE   = np.array([float(b) for b in config["parameters"]["beta"]])

# ── Publication-quality style ────────────────────────────────────────────────
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "axes.titleweight": "bold",
    "axes.labelsize": 12,
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "mathtext.fontset": "cm",
})

METHOD_ORDER = [m.upper() for m in METHODS]
REG_TYPES    = ["linear", "logistic"]
REG_LABELS   = {"linear": "Linear (OLS)", "logistic": "Logistic"}

result_dir = os.path.join("..", "results")
fig_dir    = os.path.join(result_dir, "figures_final")
os.makedirs(fig_dir, exist_ok=True)

print(f"Config loaded — {len(METHODS)} methods × {len(VAR_TYPES)} var_types")
print(f"  N:        {N_VALS}")
print(f"  p:        {P_VALS}")
print(f"  ρ:        {RHO_VALS}")
print(f"  σ²:       {SIGMA2_VALS}")
print(f"  Methods:  {METHOD_ORDER}")
print("Ready.")

In [ ]:
# ── Load both parquet files ───────────────────────────────────────────────────
df_lin = pd.read_parquet(os.path.join(result_dir, "aggregated_model_metrics.parquet"))
df_log = pd.read_parquet(os.path.join(result_dir, "aggregated_logistic_metrics.parquet"))

for d in [df_lin, df_log]:
    d["rho"]     = d["rho"].round(6)
    d["sigma_2"] = d["sigma_2"].round(6)

print(f"Linear:   {len(df_lin):,} rows × {len(df_lin.columns)} cols")
print(f"Logistic: {len(df_log):,} rows × {len(df_log.columns)} cols")

In [ ]:
# ── Derive all metrics for both DataFrames ───────────────────────────────────

_num_key = lambda c: int(c.rsplit("_", 1)[-1])

def enrich(df, label):
    """Add derived columns: sig_agreement_pct, ci_overlap_pct, coef_eucl_dist."""
    beta_od_cols = sorted([c for c in df.columns if c.startswith("beta_od_")], key=_num_key)
    beta_sd_cols = sorted([c for c in df.columns if c.startswith("beta_sd_")], key=_num_key)
    pval_od_cols = sorted([c for c in df.columns if c.startswith("pval_od_")], key=_num_key)
    pval_sd_cols = sorted([c for c in df.columns if c.startswith("pval_sd_")], key=_num_key)
    se_od_cols   = sorted([c for c in df.columns if c.startswith("se_od_")],   key=_num_key)
    se_sd_cols   = sorted([c for c in df.columns if c.startswith("se_sd_")],   key=_num_key)

    # Slope-only (exclude intercept index 0)
    beta_od_slope = [c for c in beta_od_cols if c != "beta_od_0"]
    beta_sd_slope = [c for c in beta_sd_cols if c != "beta_sd_0"]
    se_od_slope   = [c for c in se_od_cols   if c != "se_od_0"]
    se_sd_slope   = [c for c in se_sd_cols   if c != "se_sd_0"]

    # 1. Significance Agreement
    pval_od_arr = df[pval_od_cols[1:]].values
    pval_sd_arr = df[pval_sd_cols[1:]].values
    sig_od = pval_od_arr < 0.05
    sig_sd = pval_sd_arr < 0.05
    valid_mask = ~(np.isnan(pval_od_arr) & np.isnan(pval_sd_arr))
    agreement  = (sig_od == sig_sd) & valid_mask
    n_valid = valid_mask.sum(axis=1)
    n_agree = agreement.sum(axis=1)
    df["sig_agreement_pct"] = np.where(n_valid > 0, (n_agree / n_valid) * 100, np.nan)

    # 2. CI Overlap (new formula)
    bo = df[beta_od_slope].values
    bs = df[beta_sd_slope].values
    so = df[se_od_slope].values
    ss = df[se_sd_slope].values
    L_od = bo - 1.96 * so;  U_od = bo + 1.96 * so
    L_sd = bs - 1.96 * ss;  U_sd = bs + 1.96 * ss
    U_inter = np.minimum(U_od, U_sd)
    L_inter = np.maximum(L_od, L_sd)
    inter_len = np.maximum(0, U_inter - L_inter)
    W_orig = U_od - L_od
    W_syn  = U_sd - L_sd
    valid_ci = (
        ~np.isnan(bo) & ~np.isnan(bs) &
        ~np.isnan(so) & ~np.isnan(ss) &
        (so > 0) & (ss > 0) & (W_orig > 0) & (W_syn > 0)
    )
    with np.errstate(divide="ignore", invalid="ignore"):
        ci_pct = np.where(valid_ci, 0.5 * (inter_len / W_orig + inter_len / W_syn) * 100, np.nan)
    df["ci_overlap_pct"] = np.nanmean(ci_pct, axis=1)

    # 3. Coefficient Euclidean Distance
    beta_diff = df[beta_od_slope].values - df[beta_sd_slope].values
    df["coef_eucl_dist"] = np.sqrt(np.nansum(beta_diff ** 2, axis=1))

    print(f"  [{label}] Enriched — sig_agreement_pct, ci_overlap_pct, coef_eucl_dist")
    return df

df_lin = enrich(df_lin, "linear")
df_log = enrich(df_log, "logistic")

# Store in a dict keyed by regression type for easy iteration
DFS = {"linear": df_lin, "logistic": df_log}

In [ ]:
# ── Reusable 2×3 heatmap plotting function ───────────────────────────────────

def plot_2x3_heatmap(metric_col, agg_func, title_template, cmap, fmt, vmin=None, vmax=None,
                     cbar_label="", fig_prefix="plot"):
    """
    For every (var_type, N, p) combination, produce a 2-row × 3-col figure.
    Rows = Linear / Logistic.   Columns = CART / NORM / PMM.
    Each cell = 3×3 heatmap of ρ × σ².
    
    Parameters
    ----------
    metric_col : str   — column name in the enriched DataFrames.
    agg_func   : str   — pandas aggregation function ('mean', 'median', etc.).
    title_template : str — f-string template with {vt}, {N}, {p} placeholders.
    cmap : str         — matplotlib colourmap.
    fmt  : str         — annotation format, e.g. '.1f', '.3f'.
    vmin, vmax         — colour scale bounds (None = auto per figure).
    cbar_label : str   — label for the colour bar.
    fig_prefix : str   — filename prefix for saving.
    """
    for vt in VAR_TYPES:
        for N_val in N_VALS:
            for p_val in P_VALS:
                fig, axes = plt.subplots(2, 3, figsize=(18, 10), squeeze=False)

                all_vals = []
                for ri, reg in enumerate(REG_TYPES):
                    df = DFS[reg]
                    sub = df[(df["var_type"] == vt) & (df["N"] == N_val) & (df["p"] == p_val)]
                    for ci, method in enumerate(METHODS):
                        msub = sub[sub["method"] == method]
                        piv = msub.pivot_table(
                            index="rho", columns="sigma_2",
                            values=metric_col, aggfunc=agg_func,
                        )
                        piv = piv.reindex(index=RHO_VALS, columns=SIGMA2_VALS)
                        all_vals.append(piv.values)

                # Auto-scale if not given
                flat = np.concatenate([v.ravel() for v in all_vals])
                flat = flat[~np.isnan(flat)]
                lo = vmin if vmin is not None else (np.nanmin(flat) if len(flat) else 0)
                hi = vmax if vmax is not None else (np.nanmax(flat) if len(flat) else 1)

                idx = 0
                for ri, reg in enumerate(REG_TYPES):
                    for ci, method in enumerate(METHODS):
                        ax = axes[ri, ci]
                        mat = all_vals[idx]; idx += 1
                        sns.heatmap(
                            pd.DataFrame(mat, index=RHO_VALS, columns=SIGMA2_VALS),
                            ax=ax, cmap=cmap, annot=True, fmt=fmt,
                            vmin=lo, vmax=hi,
                            linewidths=0.5, linecolor="white",
                            cbar_kws={"label": cbar_label, "shrink": 0.85},
                        )
                        ax.set_xlabel(r"$\sigma^2$" if ri == 1 else "", fontsize=11)
                        ax.set_ylabel(r"$\rho$" if ci == 0 else "", fontsize=11)
                        ax.set_title(
                            f"{REG_LABELS[reg]} — {method.upper()}",
                            fontsize=12, fontweight="bold",
                        )

                fig.suptitle(
                    title_template.format(vt=vt.capitalize(), N=N_val, p=p_val),
                    fontsize=15, fontweight="bold", y=1.02,
                )
                plt.tight_layout()
                fname = f"{fig_prefix}_{vt}_N{N_val}_p{p_val}.png"
                fig.savefig(os.path.join(fig_dir, fname))
                plt.show()
                print(f"  ✓ {fname}")

print("Helper ready.")

---
## Plot 1 — Significance Agreement Rate

Percentage of predictors where OD and SD agree on significance at α = 0.05.
Higher is better (100 % = identical inferential conclusions).

In [ ]:
# ── Plot 1: Significance Agreement Rate ───────────────────────────────────────
plot_2x3_heatmap(
    metric_col="sig_agreement_pct",
    agg_func="mean",
    title_template="Significance Agreement Rate (%) — {vt}, N={N}, p={p}",
    cmap="YlGn",
    fmt=".1f",
    vmin=0, vmax=100,
    cbar_label="Agreement %",
    fig_prefix="plot1_sig_agreement",
)
print("✓ Plot 1 complete.")

---
## Plot 2 — CI Overlap (New Formula)

$$\text{CIO} = \frac{1}{2}\left(\frac{U_{\text{inter}} - L_{\text{inter}}}{U_{\text{orig}} - L_{\text{orig}}} + \frac{U_{\text{inter}} - L_{\text{inter}}}{U_{\text{syn}} - L_{\text{syn}}}\right) \times 100$$

95 % confidence intervals. 100 % = identical CIs, 0 % = completely disjoint.

In [ ]:
# ── Plot 2: CI Overlap (new formula) ─────────────────────────────────────────
plot_2x3_heatmap(
    metric_col="ci_overlap_pct",
    agg_func="mean",
    title_template="CI Overlap % (New Formula) — {vt}, N={N}, p={p}",
    cmap="YlGnBu",
    fmt=".1f",
    vmin=0, vmax=100,
    cbar_label="CI Overlap %",
    fig_prefix="plot2_ci_overlap",
)
print("✓ Plot 2 complete.")

---
## Plot 3 — Accuracy vs Pseudo R² (with Ideal Line)

Logistic: Accuracy (x) vs McFadden pseudo-R² (y) for both OD and SD.
Linear: adj R² (x) on both axes (acting as both "accuracy" and "fit quality").

The ideal diagonal shows where perfect agreement would lie.

In [ ]:
# ── Plot 3: Accuracy vs Pseudo R² / adj R² ───────────────────────────────────

rng = np.random.default_rng(42)
SAMPLE = 15_000

for vt in VAR_TYPES:
    for N_val in N_VALS:
        for p_val in P_VALS:
            fig, axes = plt.subplots(2, 3, figsize=(18, 10), squeeze=False)

            for ri, reg in enumerate(REG_TYPES):
                df = DFS[reg]
                sub = df[(df["var_type"] == vt) & (df["N"] == N_val) & (df["p"] == p_val)]

                for ci, method in enumerate(METHODS):
                    ax = axes[ri, ci]
                    msub = sub[sub["method"] == method].dropna()

                    if reg == "logistic":
                        x_od = msub["accuracy_od"].values
                        y_od = msub["pseudo_r2_od"].values
                        x_sd = msub["accuracy_sd"].values
                        y_sd = msub["pseudo_r2_sd"].values
                        xlabel = "Accuracy"
                        ylabel = "McFadden Pseudo R²"
                    else:
                        x_od = msub["adj_r2_od"].values
                        y_od = msub["adj_r2_od"].values
                        x_sd = msub["adj_r2_sd"].values
                        y_sd = msub["adj_r2_sd"].values
                        xlabel = "adj R² (OD)"
                        ylabel = "adj R² (SD)"

                    # Subsample for performance
                    n = min(SAMPLE, len(x_od))
                    idx = rng.choice(len(x_od), size=n, replace=False) if len(x_od) > 0 else []

                    if len(idx) > 0:
                        ax.scatter(x_od[idx], y_od[idx], c="#3498db", s=4, alpha=0.15,
                                   label="OD", rasterized=True)
                        ax.scatter(x_sd[idx], y_sd[idx], c="#e74c3c", s=4, alpha=0.15,
                                   label="SD", rasterized=True)

                    # Ideal diagonal
                    lims = [ax.get_xlim(), ax.get_ylim()]
                    lo = min(lims[0][0], lims[1][0])
                    hi = max(lims[0][1], lims[1][1])
                    ax.plot([lo, hi], [lo, hi], "k--", linewidth=1.5, label="Ideal")
                    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)

                    ax.set_title(f"{REG_LABELS[reg]} — {method.upper()}",
                                 fontsize=12, fontweight="bold")
                    ax.set_xlabel(xlabel if ri == 1 else "", fontsize=10)
                    ax.set_ylabel(ylabel if ci == 0 else "", fontsize=10)
                    if ri == 0 and ci == 0:
                        ax.legend(fontsize=9, markerscale=3, framealpha=0.9)

            fig.suptitle(
                f"Accuracy / R² Comparison — {vt.capitalize()}, N={N_val}, p={p_val}",
                fontsize=15, fontweight="bold", y=1.02,
            )
            plt.tight_layout()
            fname = f"plot3_accuracy_r2_{vt}_N{N_val}_p{p_val}.png"
            fig.savefig(os.path.join(fig_dir, fname))
            plt.show()
            print(f"  ✓ {fname}")

print("✓ Plot 3 complete.")

---
## Plot 4 — Deviation from True β at Each Predictor

Mean |β̂_j − β_true,j| for each predictor index j = 1…p.
OD shown as black dashed baseline; SD methods as colored solid lines.
Each subplot cell aggregates across all ρ and σ² for that (regression × method) pair.

In [ ]:
# ── Plot 4: Deviation from True Betas ────────────────────────────────────────

_num_key = lambda c: int(c.rsplit("_", 1)[-1])
true_slopes = BETA_TRUE[1:]  # β_1 … β_max

for vt in VAR_TYPES:
    for N_val in N_VALS:
        for p_val in P_VALS:
            n_slopes = min(p_val, 10)
            true_for_p = true_slopes[:n_slopes]
            beta_idx   = np.arange(1, n_slopes + 1)

            fig, axes = plt.subplots(2, 3, figsize=(18, 10), squeeze=False)

            for ri, reg in enumerate(REG_TYPES):
                df = DFS[reg]
                slope_od = sorted([c for c in df.columns if c.startswith("beta_od_") and c != "beta_od_0"], key=_num_key)[:n_slopes]
                slope_sd = sorted([c for c in df.columns if c.startswith("beta_sd_") and c != "beta_sd_0"], key=_num_key)[:n_slopes]

                for ci, method in enumerate(METHODS):
                    ax = axes[ri, ci]
                    sub = df[(df["var_type"] == vt) & (df["N"] == N_val) & (df["p"] == p_val)]

                    # OD baseline (same for all methods)
                    sub_any = sub[sub["method"] == method]
                    if not sub_any.empty:
                        od_vals = sub_any[slope_od].values
                        dev_od  = np.nanmean(np.abs(od_vals - true_for_p), axis=0)
                        ax.plot(beta_idx, dev_od, "o--", color="black", linewidth=2,
                                markersize=7, label="OD", zorder=10)

                        sd_vals = sub_any[slope_sd].values
                        dev_sd  = np.nanmean(np.abs(sd_vals - true_for_p), axis=0)
                        ax.plot(beta_idx, dev_sd, "s-", color="#e74c3c", linewidth=2,
                                markersize=7, label=f"SD ({method.upper()})", zorder=9)

                    ax.set_xlabel(r"Predictor index ($j$)" if ri == 1 else "", fontsize=10)
                    ax.set_ylabel(r"Mean $|\hat{\beta}_j - \beta_{true,j}|$" if ci == 0 else "",
                                  fontsize=10)
                    ax.set_title(f"{REG_LABELS[reg]} — {method.upper()}",
                                 fontsize=12, fontweight="bold")
                    ax.set_xticks(beta_idx)
                    ax.legend(fontsize=9, framealpha=0.9)

            fig.suptitle(
                f"Deviation from True β at Each Predictor — {vt.capitalize()}, N={N_val}, p={p_val}",
                fontsize=15, fontweight="bold", y=1.02,
            )
            plt.tight_layout()
            fname = f"plot4_beta_deviation_{vt}_N{N_val}_p{p_val}.png"
            fig.savefig(os.path.join(fig_dir, fname))
            plt.show()
            print(f"  ✓ {fname}")

print("✓ Plot 4 complete.")

---
## Plot 5 — CI Overlap % vs Coefficient Euclidean Distance

Scatter of the two fidelity metrics. Points closer to the top-left corner
(high CI overlap, low distance) indicate better synthesis quality.

In [ ]:
# ── Plot 5: CI Overlap % vs Coef Euclidean Distance ──────────────────────────

rng5 = np.random.default_rng(42)
SAMPLE5 = 8_000

for vt in VAR_TYPES:
    for N_val in N_VALS:
        for p_val in P_VALS:
            fig, axes = plt.subplots(2, 3, figsize=(18, 10), squeeze=False)

            for ri, reg in enumerate(REG_TYPES):
                df = DFS[reg]
                sub = df[(df["var_type"] == vt) & (df["N"] == N_val) & (df["p"] == p_val)]

                for ci, method in enumerate(METHODS):
                    ax = axes[ri, ci]
                    msub = sub[sub["method"] == method].dropna(subset=["ci_overlap_pct", "coef_eucl_dist"])

                    n = min(SAMPLE5, len(msub))
                    if n > 0:
                        idx = rng5.choice(len(msub), size=n, replace=False)
                        samp = msub.iloc[idx]
                        ax.scatter(
                            samp["coef_eucl_dist"], samp["ci_overlap_pct"],
                            c="#3498db", s=5, alpha=0.2, rasterized=True,
                        )

                    ax.set_xlabel(r"Coef. Euclidean Distance" if ri == 1 else "", fontsize=10)
                    ax.set_ylabel("CI Overlap %" if ci == 0 else "", fontsize=10)
                    ax.set_title(f"{REG_LABELS[reg]} — {method.upper()}",
                                 fontsize=12, fontweight="bold")

            fig.suptitle(
                f"CI Overlap % vs Coefficient Distance — {vt.capitalize()}, N={N_val}, p={p_val}",
                fontsize=15, fontweight="bold", y=1.02,
            )
            plt.tight_layout()
            fname = f"plot5_ci_vs_eucl_{vt}_N{N_val}_p{p_val}.png"
            fig.savefig(os.path.join(fig_dir, fname))
            plt.show()
            print(f"  ✓ {fname}")

print("✓ Plot 5 complete.")

---
## Plot 6 — Reproduction Fidelity: β̂_OD vs β̂_SD Scatter

Points on the diagonal indicate perfect coefficient reproduction.
Off-diagonal spread reveals systematic bias introduced by synthesis.

In [ ]:
# ── Plot 6: Reproduction Fidelity (OD vs SD coefficient scatter) ─────────────

rng6 = np.random.default_rng(42)
SAMPLE6 = 15_000

_num_key = lambda c: int(c.rsplit("_", 1)[-1])

for vt in VAR_TYPES:
    for N_val in N_VALS:
        for p_val in P_VALS:
            fig, axes = plt.subplots(2, 3, figsize=(18, 10), squeeze=False)

            for ri, reg in enumerate(REG_TYPES):
                df = DFS[reg]
                slope_od = sorted([c for c in df.columns if c.startswith("beta_od_") and c != "beta_od_0"], key=_num_key)[:p_val]
                slope_sd = sorted([c for c in df.columns if c.startswith("beta_sd_") and c != "beta_sd_0"], key=_num_key)[:p_val]
                sub = df[(df["var_type"] == vt) & (df["N"] == N_val) & (df["p"] == p_val)]

                for ci, method in enumerate(METHODS):
                    ax = axes[ri, ci]
                    msub = sub[sub["method"] == method]
                    n = min(SAMPLE6, len(msub))

                    if n > 0:
                        idx = rng6.choice(len(msub), size=n, replace=False)
                        samp = msub.iloc[idx]
                        bod = samp[slope_od].values.ravel()
                        bsd = samp[slope_sd].values.ravel()
                        keep = ~(np.isnan(bod) | np.isnan(bsd))
                        ax.scatter(bod[keep], bsd[keep], c="#3498db", s=4, alpha=0.12,
                                   rasterized=True)

                    # y = x reference
                    lims = [ax.get_xlim(), ax.get_ylim()]
                    lo = min(lims[0][0], lims[1][0])
                    hi = max(lims[0][1], lims[1][1])
                    margin = 0.05 * max(hi - lo, 0.1)
                    ref = [lo - margin, hi + margin]
                    ax.plot(ref, ref, "k--", linewidth=1.5)
                    ax.set_xlim(ref); ax.set_ylim(ref)
                    ax.set_aspect("equal", adjustable="box")

                    ax.set_title(f"{REG_LABELS[reg]} — {method.upper()}",
                                 fontsize=12, fontweight="bold")
                    ax.set_xlabel(r"$\hat{\beta}_{OD}$" if ri == 1 else "", fontsize=10)
                    ax.set_ylabel(r"$\hat{\beta}_{SD}$" if ci == 0 else "", fontsize=10)

            fig.suptitle(
                f"Reproduction Fidelity: β̂_OD vs β̂_SD — {vt.capitalize()}, N={N_val}, p={p_val}",
                fontsize=15, fontweight="bold", y=1.02,
            )
            plt.tight_layout()
            fname = f"plot6_repro_fidelity_{vt}_N{N_val}_p{p_val}.png"
            fig.savefig(os.path.join(fig_dir, fname))
            plt.show()
            print(f"  ✓ {fname}")

print("✓ Plot 6 complete.")

---
## Plot 7 — Per-Predictor Mean Absolute Bias

Heatmap where rows = predictor indices (X1…Xp) and the cell value is
mean |β̂_OD,j − β̂_SD,j|. One 2×3 figure per (var_type, N, p).

In [ ]:
# ── Plot 7: Per-Predictor Mean Absolute Bias Heatmap ─────────────────────────

_num_key = lambda c: int(c.rsplit("_", 1)[-1])

for vt in VAR_TYPES:
    for N_val in N_VALS:
        for p_val in P_VALS:
            n_pred = min(p_val, 10)

            fig, axes = plt.subplots(2, 3, figsize=(18, max(6, 0.8 * n_pred + 4)), squeeze=False)

            # Collect all values for consistent color scale
            all_mats = []

            for ri, reg in enumerate(REG_TYPES):
                df = DFS[reg]
                slope_od = sorted([c for c in df.columns if c.startswith("beta_od_") and c != "beta_od_0"], key=_num_key)[:n_pred]
                slope_sd = sorted([c for c in df.columns if c.startswith("beta_sd_") and c != "beta_sd_0"], key=_num_key)[:n_pred]
                sub = df[(df["var_type"] == vt) & (df["N"] == N_val) & (df["p"] == p_val)]

                for ci, method in enumerate(METHODS):
                    msub = sub[sub["method"] == method]
                    bod = msub[slope_od].values
                    bsd = msub[slope_sd].values
                    abs_bias = np.abs(bod - bsd)
                    mat = np.nanmean(abs_bias, axis=0).reshape(1, -1)  # (1, p)
                    all_mats.append(mat)

            # Determine global vmin/vmax
            flat = np.concatenate([m.ravel() for m in all_mats])
            flat = flat[~np.isnan(flat)]
            gmin = np.nanmin(flat) if len(flat) else 0
            gmax = np.nanmax(flat) if len(flat) else 1

            idx_m = 0
            for ri, reg in enumerate(REG_TYPES):
                for ci, method in enumerate(METHODS):
                    ax = axes[ri, ci]
                    mat = all_mats[idx_m]; idx_m += 1
                    # Transpose to have predictors as rows
                    mat_t = mat.T  # (p, 1)

                    sns.heatmap(
                        pd.DataFrame(mat_t,
                                     index=[f"X{j+1}" for j in range(n_pred)],
                                     columns=["Bias"]),
                        ax=ax, cmap="YlOrRd", annot=True, fmt=".3f",
                        vmin=gmin, vmax=gmax,
                        linewidths=0.5, linecolor="white",
                        cbar_kws={"label": r"Mean $|\hat{\beta}_{OD,j} - \hat{\beta}_{SD,j}|$", "shrink": 0.85},
                    )
                    ax.set_title(f"{REG_LABELS[reg]} — {method.upper()}",
                                 fontsize=12, fontweight="bold")
                    ax.set_ylabel("Predictor" if ci == 0 else "", fontsize=11)
                    ax.tick_params(axis="y", labelrotation=0)

            fig.suptitle(
                f"Per-Predictor Mean Absolute Bias — {vt.capitalize()}, N={N_val}, p={p_val}",
                fontsize=15, fontweight="bold", y=1.02,
            )
            plt.tight_layout()
            fname = f"plot7_predictor_bias_{vt}_N{N_val}_p{p_val}.png"
            fig.savefig(os.path.join(fig_dir, fname))
            plt.show()
            print(f"  ✓ {fname}")

print("✓ Plot 7 complete.")